# 04 — Commodities Signal Scan

Energy, metals, and softs velocity leaderboard.
No API key needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'specvel'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
sns.set_theme(style='darkgrid')

from adapters.commodities import CommoditiesAdapter
from core import compute_velocity
from leaderboard import run_leaderboard, print_leaderboard, save_leaderboard

START = '2021-01-01'
END   = '2026-03-10'

In [ ]:
adapter = CommoditiesAdapter()
df = run_leaderboard(adapter, start=START, end=END,
                     asset_class='commodities', top_n=20)
print_leaderboard(df, title='COMMODITIES VELOCITY LEADERBOARD')

## Conviction Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2ecc71' if c > 0 else ('#e74c3c' if c < 0 else '#95a5a6')
          for c in df['conviction']]
bars = ax.barh(df['label'], df['conviction'], color=colors, edgecolor='white',
               linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Conviction Score (-4 to +4)', fontsize=11)
ax.set_title('Commodities Velocity Conviction Scores', fontsize=13, fontweight='bold')
ax.set_xlim(-4.5, 4.5)
ax.invert_yaxis()

for bar, val in zip(bars, df['conviction']):
    ax.text(val + (0.1 if val >= 0 else -0.1), bar.get_y() + bar.get_height()/2,
            f'{val:+d}', va='center', ha='left' if val >= 0 else 'right',
            fontsize=9, fontweight='bold')

plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/commodities_conviction.png', dpi=150)
plt.show()

## Energy Complex — Velocity Comparison

In [ ]:
energy = ['CL=F', 'BZ=F', 'NG=F', 'RB=F']

fig, ax = plt.subplots(figsize=(13, 5))
for ticker in energy:
    try:
        raw    = adapter.fetch(ticker, START, END)
        normed = adapter.normalize(raw)
        vel    = compute_velocity(normed)
        label  = adapter.label(ticker)
        ax.plot(vel.index, vel.values, label=label, linewidth=1.3)
    except Exception as e:
        print(f'{ticker}: {e}')

ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('Energy Complex — Spectral Velocity', fontsize=12, fontweight='bold')
ax.set_ylabel('Velocity')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../results/energy_velocity.png', dpi=150)
plt.show()

In [ ]:
save_leaderboard(df, '../results/commodities_leaderboard.csv')
print('Saved.')